# 📘 Fincept Notebook — Dividend Growth & DRIP Compounding

**Investing · Intermediate · ~18 min · pandas + numpy**

A dividend reinvestment plan (DRIP) automatically uses every dividend payment to buy more shares. Those extra shares then pay their own dividends, which buy still more shares — the snowball that powers long-run equity returns. This notebook models a single dividend-growth stock over 25 years and measures exactly how much the 'reinvest' decision is worth versus taking the cash.

**What you'll learn**
- How to project shares, price, and dividend-per-share forward year by year
- How to build the full DRIP table in pandas and read yield-on-cost
- The size of the compounding gap between reinvesting and spending dividends

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. The inputs

We model one position with five assumptions: starting shares, starting price, annual price appreciation, the starting dividend yield, and how fast the dividend grows each year. Every code cell re-embeds these so it runs independently.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

# --- Model inputs (edit these) ---
START_SHARES   = 100.0    # shares owned at year 0
START_PRICE    = 50.00    # price per share at year 0 ($)
PRICE_GROWTH   = 0.06     # annual price appreciation (6%)
START_YIELD    = 0.030    # starting dividend yield (3.0% of price)
DIV_GROWTH     = 0.07     # annual dividend-per-share growth (7%)
YEARS          = 25

start_div_per_share = START_PRICE * START_YIELD
cost_basis = START_SHARES * START_PRICE

print("Starting position")
print(f"  Shares             : {START_SHARES:,.2f}")
print(f"  Price / share      : ${START_PRICE:,.2f}")
print(f"  Market value       : ${cost_basis:,.2f}")
print(f"  Dividend / share   : ${start_div_per_share:,.3f}  ({START_YIELD:.1%} yield)")
print(f"  Year-1 dividends   : ${START_SHARES * start_div_per_share:,.2f}")
print()
print("Growth assumptions")
print(f"  Price appreciation : {PRICE_GROWTH:.1%} / yr")
print(f"  Dividend growth    : {DIV_GROWTH:.1%} / yr")
print(f"  Horizon            : {YEARS} years")


## 2. Build the DRIP table

Each year: the dividend per share grows, dividends paid = shares × dividend/share, those dividends buy new shares at **that year's price**, and share count rises. We capture every year in a DataFrame and also compute **yield-on-cost** — this year's dividend income divided by the original cost basis.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

START_SHARES, START_PRICE = 100.0, 50.00
PRICE_GROWTH, START_YIELD, DIV_GROWTH, YEARS = 0.06, 0.030, 0.07, 25

cost_basis = START_SHARES * START_PRICE
div_per_share = START_PRICE * START_YIELD
shares = START_SHARES
price = START_PRICE

rows = []
for year in range(1, YEARS + 1):
    # price and dividend grow at the start of each year
    price = price * (1 + PRICE_GROWTH)
    div_per_share = div_per_share * (1 + DIV_GROWTH)

    dividends = shares * div_per_share
    new_shares = dividends / price          # reinvest at this year's price
    shares += new_shares
    value = shares * price
    yield_on_cost = (shares * div_per_share) / cost_basis

    rows.append({
        "Year": year,
        "Price": round(price, 2),
        "Div/Share": round(div_per_share, 4),
        "Dividends": round(dividends, 2),
        "NewShares": round(new_shares, 4),
        "Shares": round(shares, 4),
        "Value": round(value, 2),
        "YieldOnCost": round(yield_on_cost * 100, 2),
    })

df = pd.DataFrame(rows)
# Show first 5 and last 5 years to keep the table readable
show = pd.concat([df.head(5), df.tail(5)])
print("DRIP projection (years 1-5 and 21-25):\n")
print(show.to_string(index=False))

print(f"\nShares grew from {START_SHARES:,.1f} to {df['Shares'].iloc[-1]:,.1f} "
      f"(+{df['Shares'].iloc[-1] / START_SHARES - 1:.0%}) purely from reinvested dividends.")
print(f"Yield-on-cost climbed from {START_YIELD:.1%} to {df['YieldOnCost'].iloc[-1]:.1f}%.")


## 3. Reinvest vs take the cash

Now the headline comparison. The **WITH reinvestment** path is the DRIP above. The **WITHOUT** path holds the original shares forever and pockets each dividend as cash; we sum that cash and add it to the unchanged stock value. Same business, same dividends — only the reinvest decision differs.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

START_SHARES, START_PRICE = 100.0, 50.00
PRICE_GROWTH, START_YIELD, DIV_GROWTH, YEARS = 0.06, 0.030, 0.07, 25

cost_basis = START_SHARES * START_PRICE

# --- WITH reinvestment (DRIP) ---
shares_drip = START_SHARES
price = START_PRICE
dps = START_PRICE * START_YIELD
for _ in range(YEARS):
    price *= (1 + PRICE_GROWTH)
    dps  *= (1 + DIV_GROWTH)
    shares_drip += (shares_drip * dps) / price
drip_value = shares_drip * price

# --- WITHOUT reinvestment (cash out) ---
price = START_PRICE
dps = START_PRICE * START_YIELD
cash_collected = 0.0
for _ in range(YEARS):
    price *= (1 + PRICE_GROWTH)
    dps  *= (1 + DIV_GROWTH)
    cash_collected += START_SHARES * dps      # shares never change
stock_value_nocash = START_SHARES * price
nocash_total = stock_value_nocash + cash_collected

summary = pd.DataFrame([
    {"Strategy": "WITH reinvest (DRIP)", "FinalShares": round(shares_drip, 2),
     "StockValue": round(drip_value, 2), "CashCollected": 0.00,
     "TotalWealth": round(drip_value, 2)},
    {"Strategy": "WITHOUT (cash out)", "FinalShares": round(START_SHARES, 2),
     "StockValue": round(stock_value_nocash, 2), "CashCollected": round(cash_collected, 2),
     "TotalWealth": round(nocash_total, 2)},
])
print(summary.to_string(index=False))

gap = drip_value - nocash_total
print(f"\nStarting cost basis        : ${cost_basis:,.2f}")
print(f"Compounding advantage      : ${gap:,.2f}")
print(f"DRIP total / cash-out total: {drip_value / nocash_total:.2f}x")
print("\nReinvesting wins because each new share itself earns a growing dividend —")
print("the cash-out investor's share count (and so dividend income) never grows.")


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
